# Infant Formula Comparison: Pipeline Walkthrough

This notebook traces the full pipeline from raw PDF extraction through cleaning to visualization.
All path resolution delegates to the installed `infant_formula` package — no hardcoded paths in notebook cells.

In [37]:
import json
import pandas as pd
import plotly.express as px
import openpyxl
import re


from infant_formula.collate import INPUTS_DIR, OUTPUTS_DIR, collate
from infant_formula.extract_pdf_tables import CONFIGS_DIR

## 1. Raw Extraction

`extract_pdf_tables.py` reads each PDF page-by-page and saves one CSV per page.
The naming convention is `{pdf_stem}_p{page}_t{table_index}.csv`.

In [18]:
csvs = sorted(OUTPUTS_DIR.glob("*_p*_t*.csv"))
print(f"{len(csvs)} page-level CSV files:\n")
for f in csvs:
    df_raw = pd.read_csv(f)
    print(f"  {f.name}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")

5 page-level CSV files:

  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p1_t0.csv: 24 rows x 14 cols
  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p2_t0.csv: 22 rows x 14 cols
  Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_p3_t0.csv: 8 rows x 14 cols
  Consumer-Reports-Test-Results-Infant-Formula-v2_p1_t0.csv: 22 rows x 14 cols
  Consumer-Reports-Test-Results-Infant-Formula-v2_p2_t0.csv: 19 rows x 14 cols


In [19]:
# Peek at the first page's raw extracted content
pd.read_csv(csvs[0]).head(3)

,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year
0,POWDERED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026
1,Enfamil,Sensitive,10.6,9.0,1.8,ND,ND,272,"8,026,667",ND,ND,ND,ND,2026
2,Enfamil,Nutramigen with Probiotic\nLGG,8.4,7.6,5.7,2.1,ND,"2,187","6,970,000",8,ND,ND,12.0,2026


In [20]:
# Peek at the last page's raw extracted content to confirm that spaces are removed from brand names
pd.read_csv(csvs[0]).tail(6)

,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year
18,Mama Bear,Sensitivity Premium,3.6,NT,1.5,2.4,ND,"2,823","6,200,000",ND,ND,ND,ND,2026
19,Jovie,Stage 1 Organic Goat Milk,2.4,NT,1.9,ND,ND,706,"6,800,000",ND,ND,ND,ND,2026
20,Pure Bliss by\nSimilac,Irish Farms,3.5,NT,2.0,ND,0.7,587,"6,396,667",ND,ND,ND,ND,2026
21,Pure Goat,Bio Complete,ND,NT,2.1,2.5,ND,339,"6,790,000",ND,ND,ND,ND,2026
22,HiPP,Hypoallergenic,2.6,NT,1.9,3.3,ND,530,"6,300,000",ND,ND,ND,ND,2026
23,Holle,Organic Goat Milk,2.1,NT,2.1,3.3,ND,206,"6,500,000",ND,ND,ND,ND,2026


## 2. Column Configuration

The PDFs have rotated column headers that pdfplumber reads top-to-bottom (reversing each character).
`inspect_header_row()` decodes them and writes a JSON template to `configs/` for human review.
The final column names used in extraction come from these edited config files.

In [21]:
for cfg in sorted(CONFIGS_DIR.glob("*_columns.json")):
    if "original" in cfg.name:
        continue
    cols = json.loads(cfg.read_text())
    print(f"{cfg.name}:")
    for i, col in enumerate(cols):
        print(f"  {i:2d}: {col}")
    print()

Consumer-Reports-Baby-Formula-Test-Results-v4-March-2026_columns.json:
   0: brand
   1: model
   2: total_arsenic
   3: inorganic_arsenic
   4: lead
   5: cadmium
   6: mercury
   7: aluminum
   8: potassium
   9: bisphenol_a
  10: bisphenol_f
  11: bisphenol_s
  12: acrylamide

Consumer-Reports-Test-Results-Infant-Formula-v2_columns.json:
   0: brand
   1: model
   2: total_arsenic
   3: inorganic_arsenic
   4: lead
   5: cadmium
   6: mercury
   7: aluminum
   8: potassium
   9: bisphenol_a
  10: bisphenol_f
  11: bisphenol_s
  12: acrylamide



## 3. Collation & Cleaning

`collate()` does three things:
1. Concatenates all per-page CSVs into one dataframe
2. Strips thousands-separator commas from numeric strings (e.g. `"8,026,667"` → `"8026667"`)
3. Drops subheader rows (e.g. `"POWDERED"`, `"LIQUID"`) that have no `model` value

The cell below captures a raw value before `collate()` runs, then checks the cleaned version after.

In [22]:
# Skip subheader rows (e.g. "POWDERED") — they have no model value and are dropped by collate()
first_page = pd.read_csv(csvs[0])
first_data_row = first_page[first_page["model"].notna()].iloc[0]
sample_brand = first_data_row["brand"]
raw_potassium = first_data_row["potassium"]
print(f"Raw (from page CSV)  — brand: {sample_brand!r}, potassium: {raw_potassium!r}")

Raw (from page CSV)  — brand: 'Enfamil', potassium: '8,026,667'


In [23]:
collate()

collated = pd.read_csv(OUTPUTS_DIR / "all_formula.csv")
cleaned_row = collated[collated["brand"] == sample_brand].iloc[0]
print(f"After collate()      — brand: {cleaned_row['brand']!r}, potassium: {cleaned_row['potassium']!r}")
print(f"\nCollated shape: {collated.shape}")
collated.head(3)

Saved 90 rows to /Users/sarahfouzia/llm_workspace/repos/infant_formula_comparison/outputs/all_formula.csv
After collate()      — brand: 'Enfamil', potassium: '8,026,667'

Collated shape: (90, 14)


,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year
0,Enfamil,Sensitive,10.6,9.0,1.8,ND,ND,272,"8,026,667",ND,ND,ND,ND,2026
1,Enfamil,Nutramigen with Probiotic\nLGG,8.4,7.6,5.7,2.1,ND,2187,"6,970,000",8,ND,ND,12.0,2026
2,Dr. Brown’s,Good Start Soy-Ease Pro,9.9,7.9,2.3,4.3,0.5,4403,"6,806,667",ND,ND,ND,ND,2026


## 4. Input table enriched with website_link and ingredient_list

`all_formula_ingredients_manual_entry.csv`is a table with two additional columns for 'website_link' and 'ingredient_list'

Added those two columns manually as different brands render their ingredient lists differently

Using df for the output from this extraction and df_website_ingredients for the file with the links i.e. `all_formula_ingredients_manual_entry.csv`

The main reason for doing this is to keep the extraction step separate from the manual entry step. For example, if there are additional formulas tested next year, it would be better to join the existing df_website_ingredients with the additional entries from the extraction step in df. The current dataset has the chemical analyses from 2025 and 2026

In [24]:
df_website_ingredients = pd.read_excel(INPUTS_DIR / "all_formula_ingredients_manual_entry.xlsx")
print(df_website_ingredients.head(15))
print(df_website_ingredients.tail(15))

    index_100                 brand                                model  \
0         101                    A2                             Platinum   
1         102               Aptamil                    First Infant Milk   
2         103  Baby’s Only\nOrganic                   Complete Nutrition   
3         104                Bobbie  Grass-Fed Whole Milk\n(non-organic)   
4         105                Bobbie                              Organic   
5         106                Bobbie                       Organic Gentle   
6         107                Bobbie                   Organic Whole Milk   
7         108                  Bubs       Stage 1 Easy-digest\nGoat Milk   
8         109                  Bubs           Stage 1 Organic\nGrass Fed   
9         110               ByHeart                      Whole Nutrition   
10        111           Dr. Brown’s                            GentlePro   
11        112           Dr. Brown’s              Good Start Soy-Ease Pro   
12        11

In [25]:
# conversion from xlsx file to csv during manual entry didn't work
# df_website_ingredients = pd.read_csv(INPUTS_DIR / "all_formula_ingredients_manual_entry.csv")
# print(f"Shape: {df_website_ingredients.shape}\n")
# df_website_ingredients.dtypes

In [42]:
df = pd.read_csv(OUTPUTS_DIR / "all_formula.csv")
print(f"Shape: {df.shape}\n")
df.dtypes

Shape: (90, 14)



brand                  str
model                  str
total_arsenic          str
inorganic_arsenic      str
lead                   str
cadmium                str
mercury                str
aluminum               str
potassium              str
bisphenol_a            str
bisphenol_f            str
bisphenol_s            str
acrylamide             str
report_year          int64
dtype: object

## 5. Left Join df with df_website_ingredients on brand and model

In [43]:
df = df.merge(
      df_website_ingredients[["brand", "model", "report_year", "website_link", "ingredient_list"]],
      on=["brand", "model", "report_year"],
      how="left",
  ).sort_values(["brand","model"]).reset_index(drop=True)

In [44]:
df.head()
df.tail()

,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year,website_link,ingredient_list
85,Sprout Organics,Plant-Based,15.1,4.5,8.5,22.4,1.6,4463,"5,166,667",3,ND,ND,18,2026,https://sproutorganic.com.au/products/infant-formula,"Organic Hydrolysed Rice Starch, Organic Oil Blend (Organic Coconut Oil, Organic Canola Oil, Organic Sunflower Oil, emulsifier organic sunflower lecithin, antioxidant natural mixed tocopherols), Sprout Organic Protein Blend (Organic Sprouted Rice Protein, Organic Sprouted Pea Protein), Docosahexaenoic Acid (DHA from algae), Arachidonic Acid (ARA from fungal), Vitamins, Minerals & Amino Acid Mix (Potassium, Calcium, Chloride, Phosphorus, L-Lysine, Choline, Sodium, Inositol, Vitamin C, L-Tryptophan, L-Threonine, Magnesium, Iron, Niacin, L-carnitine, Zinc, Vitamin E, Pantothenic Acid, Vitamin A, Riboflavin, Copper, Thiamin, Vitamin B6, Folic Acid, Manganese, Vitamin K1, Iodine, Biotin, Selenium, Vitamin D2, Vitamin B12)."
86,Up&Up (Target),Hypoallergenic,7.4,5.9,2.1,ND,ND,448,"7,673,333",ND,ND,ND,ND,2026,https://www.target.com/p/non-gmo-hypoallergenic-powder-infant-formula-27-8oz-up-38-up-8482/-/A-85428762,"corn syrup solids, vegetable oils (palm olein, soy, coconut, high oleic (safflower or sunflower) oil), casein hydrolysate (from milk)+, modified corn starch, less than 2%: mortierella alpina oil, schizochytrium sp. oil, lactobacillus rhamnosus, beta-carotene, vitamin a palmitate, vitamin d3, vitamin e acetate, vitamin k1, thiamin hydrochloride, riboflavin, vitamin b6 hydrochloride, vitamin b12, niacinamide, folic acid, calcium pantothenate, biotin, ascorbic acid, choline bitartrate, inositol, calcium citrate, calcium hydroxide, calcium phosphate, magnesium chloride, ferrous sulfate, zinc sulfate, manganese sulfate, cupric sulfate, potassium iodide, sodium selenite, sodium citrate, potassium citrate, potassium chloride, potassium bicarbonate, l-cystine, l-tryptophan, taurine, l-carnitine, calcium carbonate, calcium chloride, potassium hydroxide, potassium phosphate, ascorbyl palmitate, mixed tocopherol concentrate. ‡a source of hypoallergenic protein *a source of arachidonic acid (ara) **a source of docosahexaenoic acid (dha)"
87,Up&Up (Target),Premium,ND,NT,ND,ND,ND,191,"7,005,000",ND,ND,ND,ND,2025,https://www.target.com/p/premium-powder-infant-formula-22-2oz-up-38-up-8482/-/A-87841478,"nonfat milk, lactose, vegetable oils (palm olein, soy, coconut, high oleic [safflower or sunflower] oil), whey protein concentrate, less than 1%: 2'-fucosyllactose (2'-fl)◊, mortierella alpina oil*, schizochytrium sp. oil**, soy lecithin, vitamin a palmitate, vitamin d3, vitamin e acetate, vitamin k ,, thiamine hydrochloride, riboflavin, vitamin b; hydrochloride, vitamin b12, niacinamide, folic acid, calcium pantothenate, biotin, ascorbic acid, choline bitartrate, inositol, calcium carbonate, calcium chloride, calcium hydroxide, magnesium chloride, ferrous sulfate, zinc sulfate, manganese sulfate, cupric sulfate, potassium bicarbonate, potassium iodide, potassium hydroxide, potassium phosphate, sodium selenite, sodium citrate, taurine, l-carnitine, beta-carotene, mixed tocopherol concentrate, ascorbyl palmitate, monoglycerides, nucleotides (cytidine-5'-monophosphate, disodium uridine-5'-monophosphate, adenosine-5'-monophosphate, disodium guanosine-5'-monophosphate)."
88,Up&Up (Target),Sensitivity Premium,5.7,5.0,1.8,ND,ND,1585,"5,593,333",ND,ND,ND,ND,2026,https://www.target.com/p/sensitivity-premium-infant-formula-with-iron-powder-22-5oz-up-38-up-8482/-/A-85428763,"corn syrup, milk protein isolate, high oleic (safflower or sunflower) oil, sucrose, soy oil, coconut oil, less than 2%: mortierella alpina oil *, schizochytrium sp. oil **, 2'-fucosyllactose (2'-fl)º, lacto-n-neotetraose (lnnt)0, lutein, calcium phosphate, potassium citrate, potassium chloride, sodium citrate, magnesium phosphate, ascorbic acid, calcium carbonate, ferrous sulfa

In [45]:
df_xlsx = pd.read_excel(INPUTS_DIR / "all_formula_ingredients_manual_entry.xlsx")
df_xlsx.dtypes

index_100              int64
brand                    str
model                    str
website_link             str
ingredient_list          str
has_soy              float64
has_palm_oil         float64
total_arsenic         object
inorganic_arsenic     object
lead                  object
cadmium               object
mercury               object
aluminum              object
potassium              int64
bisphenol_a           object
bisphenol_f              str
bisphenol_s              str
acrylamide            object
report_year            int64
dtype: object

## 6. Add an index starting from 100 on the left
This makes it easier to find the records in alphabetical order

In [46]:
df.insert(0,'index_100', range(101, 101+ len(df)))

In [47]:
df.head()

,index_100,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year,website_link,ingredient_list
0,101,A2,Platinum,ND,NT,2.9,ND,ND,1770,"5,580,000",ND,ND,ND,ND,2025,https://a2platinum.com/pages/why-a2,recalled
1,102,Aptamil,First Infant Milk,ND,NT,1.5,ND,ND,215,"5,670,000",ND,ND,ND,ND,2025,https://www.nutricia.co.uk/hcp/pim-products/aptamil-first-infant-milk-with-pronutra-powder.html,"Dairy-based blend (of which 29% is fermented) [Lactose (from milk), Vegetable oils (High oleic sunflower oil, Coconut oil, Rapeseed oil, Sunflower oil), Skimmed milk, Demineralised whey (from milk), Whey concentrate (from milk), Fish oil, Potassium citrate, Calcium carbonate, Magnesium chloride, Choline chloride, Calcium phosphate, Sodium citrate, Potassium chloride, Vitamin C, Emulsifier (Soy lecithin), Inositol, Antioxidant (Vitamin C), Pantothenic acid, Nicotinamide, Vitamin E, Thiamin, Riboflavin, Vitamin B₆, Potassium iodide, Folic acid, Vitamin K₁, Biotin, Vitamin B₁₂], Galacto-oligosaccharides (GOS) (from milk), Milk protein, Fructo-oligosaccharides (FOS), Oil from Mortierella Alpina, Sodium chloride, L-Tryptophan, Ferrous sulphate, Zinc sulphate, L-Carnitine, Copper sulphate, Vitamin A, Sodium selenite, Manganese sulphate, Vitamin D₃. Allergy Advice: For allergens, see ingredients in bold."
2,103,Baby’s Only\nOrganic,Complete Nutrition,ND,NT,1.4,ND,ND,307,"6,080,000",ND,ND,ND,ND,2025,https://babysonly.com/collections/shop-all/products/organic-gentle-infant-formula,"Organic Lactose, Organic Nonfat Milk (Source Of A2 Proteins), Organic High Oleic Sunflower, Organic Soybean Oil, Organic Whey Protein Concentrate, Organic Coconut Oil, Less Than 2% Of: Organic Soy Lecithin, Calcium Carbonate, Calcium Phosphate, Sodium Chloride, Potassium Chloride, Potassium Citrate, Magnesium Phosphate, Choline Bitartrate, Calcium Ascorbate, Inositol, Ferrous Sulfate, D-Alpha Tocopheryl Acetate, Zinc Sulfate, Niacinamide, Calcium Pantothenate, Biotin, Vitamin A, Thiamin Hydrochloride, Copper Sulfate, Riboflavin, Pyridoxine Hydrochloride, Sodium Selenate, Manganese Sulfate, Phylloquinone, Folic Acid, Vitamin D3, And Vitamin B12."
3,104,Bobbie,Grass-Fed Whole Milk\n(non-organic),ND,NT,ND,ND,ND,NaN,"7,080,000",ND,ND,ND,ND,2026,https://www.hibobbie.com/products/grass-fed-whole-milk-infant-formula-14-1oz?variant=42520514101333,"lactose, organic whole milk, organic linoleic sunflower oil, organic low erucic acid rapeseed oil, organic whey protein concentrate, organic high oleic sunflower oil, organic coconut oil, less than 2 % of: organic low erucic acid rapeseed lecithin, schizochytrium sp. oil*, mortierella alpina oil**, calcium carbonate, calcium phosphate, potassium chloride, sodium chloride, potassium citrate, l-carnitine, magnesium phosphate, calcium ascorbate, choline bitartrate, inositol, potassium iodide, ferrous sulfate, d-alpha tocopheryl acetate, zinc sulfate, vitamin c palmitate, niacinamide, calcium pantothenate, biotin, vitamin a, thiamin hydrochloride, copper sulfate, riboflavin, mixed tocopherols, pyridoxine hydrochloride, sodium selenate, manganese sulfate, phylloquinone, folic acid, vitamin d3, vitamin b12. * a source of docosahexaenoic acid (dha) ** a source of arachidonic acid (ara)"
4,105,Bobbie,Organic,ND,NT,1.2,ND,ND,1030,"6,740,000",ND,ND,ND,ND,2025,https://www.hibobbie.com/products/bobbie-organic-infant-formula?variant=42520534450261,"organic lactose, organic nonfat milk, organic high oleic (safflower or sunflower) oil, organic low erucic acid rapeseed oil, organic whey protein concentrate, organic coconut oil, organic linoleic (sunflower or safflower) oil, less than 1%: organic low erucic acid rapeseed lecithin, schizochytrium sp. oil; mortierella alpina oil;"" calcium phosphate, potassium citrate, sodium chloride, calcium carbonate, potassium hydroxide, potassium phosphate, magnesium chloride, potassium bicarbonate, ferrous sulfate, potassium chloride, zi

In [32]:
df.tail()

,index_100,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year,website_link,ingredient_list
85,186,Sprout Organics,Plant-Based,15.1,4.5,8.5,22.4,1.6,4463,"5,166,667",3,ND,ND,18,2026,https://sproutorganic.com.au/products/infant-formula,"Organic Hydrolysed Rice Starch, Organic Oil Blend (Organic Coconut Oil, Organic Canola Oil, Organic Sunflower Oil, emulsifier organic sunflower lecithin, antioxidant natural mixed tocopherols), Sprout Organic Protein Blend (Organic Sprouted Rice Protein, Organic Sprouted Pea Protein), Docosahexaenoic Acid (DHA from algae), Arachidonic Acid (ARA from fungal), Vitamins, Minerals & Amino Acid Mix (Potassium, Calcium, Chloride, Phosphorus, L-Lysine, Choline, Sodium, Inositol, Vitamin C, L-Tryptophan, L-Threonine, Magnesium, Iron, Niacin, L-carnitine, Zinc, Vitamin E, Pantothenic Acid, Vitamin A, Riboflavin, Copper, Thiamin, Vitamin B6, Folic Acid, Manganese, Vitamin K1, Iodine, Biotin, Selenium, Vitamin D2, Vitamin B12)."
86,187,Up&Up (Target),Hypoallergenic,7.4,5.9,2.1,ND,ND,448,"7,673,333",ND,ND,ND,ND,2026,https://www.target.com/p/non-gmo-hypoallergenic-powder-infant-formula-27-8oz-up-38-up-8482/-/A-85428762,"corn syrup solids, vegetable oils (palm olein, soy, coconut, high oleic (safflower or sunflower) oil), casein hydrolysate (from milk)+, modified corn starch, less than 2%: mortierella alpina oil, schizochytrium sp. oil, lactobacillus rhamnosus, beta-carotene, vitamin a palmitate, vitamin d3, vitamin e acetate, vitamin k1, thiamin hydrochloride, riboflavin, vitamin b6 hydrochloride, vitamin b12, niacinamide, folic acid, calcium pantothenate, biotin, ascorbic acid, choline bitartrate, inositol, calcium citrate, calcium hydroxide, calcium phosphate, magnesium chloride, ferrous sulfate, zinc sulfate, manganese sulfate, cupric sulfate, potassium iodide, sodium selenite, sodium citrate, potassium citrate, potassium chloride, potassium bicarbonate, l-cystine, l-tryptophan, taurine, l-carnitine, calcium carbonate, calcium chloride, potassium hydroxide, potassium phosphate, ascorbyl palmitate, mixed tocopherol concentrate. ‡a source of hypoallergenic protein *a source of arachidonic acid (ara) **a source of docosahexaenoic acid (dha)"
87,188,Up&Up (Target),Premium,ND,NT,ND,ND,ND,191,"7,005,000",ND,ND,ND,ND,2025,https://www.target.com/p/premium-powder-infant-formula-22-2oz-up-38-up-8482/-/A-87841478,"nonfat milk, lactose, vegetable oils (palm olein, soy, coconut, high oleic [safflower or sunflower] oil), whey protein concentrate, less than 1%: 2'-fucosyllactose (2'-fl)◊, mortierella alpina oil*, schizochytrium sp. oil**, soy lecithin, vitamin a palmitate, vitamin d3, vitamin e acetate, vitamin k ,, thiamine hydrochloride, riboflavin, vitamin b; hydrochloride, vitamin b12, niacinamide, folic acid, calcium pantothenate, biotin, ascorbic acid, choline bitartrate, inositol, calcium carbonate, calcium chloride, calcium hydroxide, magnesium chloride, ferrous sulfate, zinc sulfate, manganese sulfate, cupric sulfate, potassium bicarbonate, potassium iodide, potassium hydroxide, potassium phosphate, sodium selenite, sodium citrate, taurine, l-carnitine, beta-carotene, mixed tocopherol concentrate, ascorbyl palmitate, monoglycerides, nucleotides (cytidine-5'-monophosphate, disodium uridine-5'-monophosphate, adenosine-5'-monophosphate, disodium guanosine-5'-monophosphate)."
88,189,Up&Up (Target),Sensitivity Premium,5.7,5.0,1.8,ND,ND,1585,"5,593,333",ND,ND,ND,ND,2026,https://www.target.com/p/sensitivity-premium-infant-formula-with-iron-powder-22-5oz-up-38-up-8482/-/A-85428763,"corn syrup, milk protein isolate, high oleic (safflower or sunflower) oil, sucrose, soy oil, coconut oil, less than 2%: mortierella alpina oil *, schizochytrium sp. oil **, 2'-fucosyllactose (2'-fl)º, lacto-n-neotetraose (lnnt)0, lutein, calcium phosphate, potassium citrate, potassium chloride, sodium citrate, magnesium phosphate, ascorbic acid, calciu

## 7. add columns for palm_words, soy_words, has_palm_oil, has_soy

In [48]:
df = df.assign(
    palm_words=pd.Series(dtype="object"),
    soy_words=pd.Series(dtype="object"),
    has_palm_oil=pd.Series(dtype="boolean"),
    has_soy=pd.Series(dtype="boolean"),
)

In [34]:
pd.set_option("display.max_colwidth", None)
display(df.ingredient_list)

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

## 8. Check the length of each entry in ingredient_list to ensure that the ingredients are correctly parsed

In [35]:
for i in range(len(df.ingredient_list)):
    print(df.brand[i],df.model[i],len(df.ingredient_list[i]))

A2 Platinum 8
Aptamil First Infant Milk 910
Baby’s Only
Organic Complete Nutrition 652
Bobbie Grass-Fed Whole Milk
(non-organic) 891
Bobbie Organic 1053
Bobbie Organic Gentle 1155
Bobbie Organic Whole Milk 819
Bubs Stage 1 Easy-digest
Goat Milk 918
Bubs Stage 1 Organic
Grass Fed 835
ByHeart Whole Nutrition 1084
Dr. Brown’s GentlePro 1004
Dr. Brown’s Good Start Soy-Ease Pro 931
Dr. Brown’s SoothePro 1019
Earth’s Best Organic Dairy 1156
Earth’s Best Organic Sensitivity 1154
EleCare Hypoallergenic 976
Enfamil 24 Ready to Feed 980
Enfamil A.R. 730
Enfamil Concentrated 936
Enfamil Enspire Optimum 774
Enfamil Gentlease 739
Enfamil Infant Formula 858
Enfamil NeuroPro 781
Enfamil NeuroPro EnfaCare Ready
to Feed 905
Enfamil NeuroPro Enfacare 883
Enfamil NeuroPro Gentlease 813
Enfamil NeuroPro Gentlease Ready
to Feed 921
Enfamil NeuroPro Ready to Feed 845
Enfamil Nutramigen 822
Enfamil Nutramigen Ready to Feed 776
Enfamil Nutramigen with Probiotic
LGG 822
Enfamil Optimum Ready to Feed 825
Enfami

# 9. Get the list of words in the ingredient_list that have 'palm' or 'soy' for classification. 

We will use this list to identify if the formula has palm or soy ingredients. A simple way to do so is to split on commas and parentheses

In [55]:
def extract_terms(text):
    """
    Split ingredient text on commas and parentheses/brackets.
    Returns a list of cleaned terms.
    """

    if not isinstance(text, str):
        return []

    terms = re.split(r"[,()\[\]]", text)

    return [
        term.strip(" .;:").lower()
        for term in terms
        if term.strip()
    ]

In [56]:
df["ingredient_array"] = df["ingredient_list"].apply(extract_terms)

In [57]:
df.head()

,index_100,brand,model,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,...,bisphenol_s,acrylamide,report_year,website_link,ingredient_list,palm_words,soy_words,has_palm_oil,has_soy,ingredient_array
0,101,A2,Platinum,ND,NT,2.9,ND,ND,1770,"5,580,000",...,ND,ND,2025,https://a2platinum.com/pages/why-a2,recalled,NaN,NaN,<NA>,<NA>,[recalled]
1,102,Aptamil,First Infant Milk,ND,NT,1.5,ND,ND,215,"5,670,000",...,ND,ND,2025,https://www.nutricia.co.uk/hcp/pim-products/aptamil-first-infant-milk-with-pronutra-powder.html,"Dairy-based blend (of which 29% is fermented) [Lactose (from milk), Vegetable oils (High oleic sunflower oil, Coconut oil, Rapeseed oil, Sunflower oil), Skimmed milk, Demineralised whey (from milk), Whey concentrate (from milk), Fish oil, Potassium citrate, Calcium carbonate, Magnesium chloride, Choline chloride, Calcium phosphate, Sodium citrate, Potassium chloride, Vitamin C, Emulsifier (Soy lecithin), Inositol, Antioxidant (Vitamin C), Pantothenic acid, Nicotinamide, Vitamin E, Thiamin, Riboflavin, Vitamin B₆, Potassium iodide, Folic acid, Vitamin K₁, Biotin, Vitamin B₁₂], Galacto-oligosaccharides (GOS) (from milk), Milk protein, Fructo-oligosaccharides (FOS), Oil from Mortierella Alpina, Sodium chloride, L-Tryptophan, Ferrous sulphate, Zinc sulphate, L-Carnitine, Copper sulphate, Vitamin A, Sodium selenite, Manganese sulphate, Vitamin D₃. Allergy Advice: For allergens, see ingredients in bold.",NaN,NaN,<NA>,<NA>,"[dairy-based blend, of which 29% is fermented, lactose, from milk, vegetable oils, high oleic sunflower oil, coconut oil, rapeseed oil, sunflower oil, skimmed milk, demineralised whey, from milk, whey concentrate, from milk, fish oil, potassium citrate, calcium carbonate, magnesium chloride, choline chloride, calcium phosphate, sodium citrate, potassium chloride, vitamin c, emulsifier, soy lecithin, inositol, antioxidant, vitamin c, pantothenic acid, nicotinamide, vitamin e, thiamin, riboflavin, vitamin b₆, potassium iodide, folic acid, vitamin k₁, biotin, vitamin b₁₂, galacto-oligosaccharides, gos, from milk, milk protein, fructo-oligosaccharides, fos, oil from mortierella alpina, sodium chloride, l-tryptophan, ferrous sulphate, zinc sulphate, l-carnitine, copper sulphate, vitamin a, sodium selenite, manganese sulphate, vitamin d₃. allergy advice: for allergens, see ingredients in bold]"
2,103,Baby’s Only\nOrganic,Complete Nutrition,ND,NT,1.4,ND,ND,307,"6,080,000",...,ND,ND,2025,https://babysonly.com/collections/shop-all/products/organic-gentle-infant-formula,"Organic Lactose, Organic Nonfat Milk (Source Of A2 Proteins), Organic High Oleic Sunflower, Organic Soybean Oil, Organic Whey Protein Concentrate, Organic Coconut Oil, Less Than 2% Of: Organic Soy Lecithin, Calcium Carbonate, Calcium Phosphate, Sodium Chloride, Potassium Chloride, Potassium Citrate, Magnesium Phosphate, Choline Bitartrate, Calcium Ascorbate, Inositol, Ferrous Sulfate, D-Alpha Tocopheryl Acetate, Zinc Sulfate, Niacinamide, Calcium Pantothenate, Biotin, Vitamin A, Thiamin Hydrochloride, Copper Sulfate, Riboflavin, Pyridoxine Hydrochloride, Sodium Selenate, Manganese Sulfate, Phylloquinone, Folic Acid, Vitamin D3, And Vitamin B12.",NaN,NaN,<NA>,<NA>,"[organic lactose, organic nonfat milk, source of a2 proteins, organic high oleic sunflower, organic soybean oil, organic whey protein concentrate, organic coconut oil, less than 2% of: organic soy lecithin, calcium carbonate, calcium phosphate, sodium chloride, potassium chloride, potassium citrate, magnesium phosphate, choline bitartrate, calcium ascorbate, inositol, ferrous sulfate, d-alpha tocopheryl acetate, zinc sulfate, niacinamide, calcium pantothenate, biotin, vitamin a, thiamin hydrochloride, copper sulfate, riboflavin, pyridoxine hydrochloride, sodium selenate, manganese sulfate, phylloquinone, folic acid, vitamin d3, and vitamin b12]"
3,104,Bobbie,Grass-Fed Whole Milk\n(non-organic),ND,NT,ND,ND,ND,NaN,"7,080,000",...,ND,ND,2026,https://www.hibobbie.com/produc

# 10. Test logic to see how how it handles parentheses and commas in the ingredient_list
Using Aptamil (index = 1, i.e. iloc[1] as an example)

In [59]:
print(el for el in df.ingredient_array.iloc[1])

<generator object <genexpr> at 0x149abe330>


None

In [60]:
for el in df["ingredient_array"].iloc[1]:
    print(el)

dairy-based blend
of which 29% is fermented
lactose
from milk
vegetable oils
high oleic sunflower oil
coconut oil
rapeseed oil
sunflower oil
skimmed milk
demineralised whey
from milk
whey concentrate
from milk
 fish oil
potassium citrate
calcium carbonate
magnesium chloride
choline chloride
calcium phosphate
sodium citrate
potassium chloride
vitamin c
emulsifier
soy lecithin
inositol
antioxidant
vitamin c
pantothenic acid
nicotinamide
vitamin e
thiamin
riboflavin
vitamin b₆
potassium iodide
folic acid
vitamin k₁
biotin
vitamin b₁₂
galacto-oligosaccharides
gos
from milk
 milk protein
fructo-oligosaccharides
fos
oil from mortierella alpina
sodium chloride
l-tryptophan
ferrous sulphate
zinc sulphate
l-carnitine
copper sulphate
vitamin a
sodium selenite
manganese sulphate
vitamin d₃. allergy advice: for allergens
see ingredients in bold


# 11. Identify ingredient words with "palm" and "soy" to ascertain if the formula has soy or palm oil in it.

In [65]:
df["soy_words"] = df["ingredient_array"].apply(
    lambda ingredients: [
        item for item in ingredients
        if "soy" in item.lower()
    ]
)

In [67]:
df["palm_words"] = df["ingredient_array"].apply(
    lambda ingredients: [
        item for item in ingredients
        if "palm" in item.lower()
    ]
)

In [70]:
palm_counts = (
    df["palm_words"]
    .explode()              # turns lists into individual rows
    .dropna()
    .value_counts()
)

print(palm_counts)

palm_words
vitamin a palmitate                                      53
ascorbyl palmitate                                       36
contains one or more of the following: palm olein oil    12
palm olein                                               11
 vitamin a palmitate                                      7
palm oil                                                  3
vitamin c palmitate                                       2
organic palm oil                                          2
palm kernel and/or coconut oil                            2
and high 2-palmitic vegetable oil                         2
sustainable palm olein                                    1
organic palm or palm olein                                1
organic palm oil or palm olein                            1
palm olein oil                                            1
retinyl palmitate                                         1
organic palm olein or palm oil                            1
palm oil*³                   

In [72]:
palm_counts.dtypes

dtype('int64')

In [74]:
soy_counts = (
    df["soy_words"]
    .explode()              # turns lists into individual rows
    .dropna()
    .value_counts()
)

print(soy_counts)

soy_words
soy oil                                                                                   43
soy lecithin                                                                              34
soy                                                                                       15
 soy lecithin                                                                              6
soy protein isolate                                                                        5
organic soy lecithin                                                                       5
 soy protein isolate                                                                       2
organic soy oil                                                                            2
contains milk and soy. ◊ sources of prebiotic *a source of arachidonic acid                2
soy lecithin                                                                               1
organic soybean oil                                         

# 12. Get list of of words to exclude from the list of ingredients containing palm oil as false positives. Use this to build out s
E.g. ascorbyl palmitate is a source of vitamin C. It's not a palm oil containing ingredient like palm olein. So we need to exclude ingredients containing these words so we don't incorrectly mark these formulas as containing palm oil.

In [84]:
exclude_terms = [
    "vitamin",
    "ascorb", ## vitamin C is ascorbic acid, e.g. added as ascorbyl palmitate
    "retin", ## vitamin A is retinol, e.g. added as retinyl palmitate
]

for word, count in palm_counts.items():

    flag = int(
        not any(term in word for term in exclude_terms)
    )

    print(word, flag)

vitamin a palmitate 0
ascorbyl palmitate 0
contains one or more of the following: palm olein oil 1
palm olein 1
 vitamin a palmitate 0
palm oil 1
vitamin c palmitate 0
organic palm oil 1
palm kernel and/or coconut oil 1
and high 2-palmitic vegetable oil 1
sustainable palm olein 1
organic palm or palm olein 1
organic palm oil or palm olein 1
palm olein oil 1
retinyl palmitate 0
organic palm olein or palm oil 1
palm oil*³ 1
high 2-palmitic acid vegetable oil 1
vitamin a palmitate* 0
palm 1
l-ascorbyl 6-palmitate 0
